<a href="https://colab.research.google.com/github/DavidReveloLuna/MaskRCNN_Video/blob/master/CustomClasses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ⚙️ Paso 0 — Compatibilidad con Python 3.12 / TensorFlow 2.16+ / Keras 3

Mask R-CNN es una librería legacy que requiere tres ajustes críticos:

1. **Motor Keras Legacy (`tf-keras`)** — Keras 3 no es compatible con el código antiguo. Instalamos `tf-keras` y forzamos su uso con `TF_USE_LEGACY_KERAS=1`.
2. **Desactivación de Eager Execution** — Mask R-CNN usa tensores simbólicos de TF 1.x. Debemos deshabilitar eager execution antes de importar `mrcnn`.
3. **Script externo para entrenamiento** — El entrenamiento se delega a `train_mrcnn.py` para evitar conflictos de sesión/memoria en celdas interactivas.

In [ ]:
# PUNTO CRÍTICO 1 — Instalar el motor Keras legacy ANTES de cualquier import de TF/Keras
# tf-keras expone la API de Keras 2 bajo tf.keras para librerías antiguas
!pip install -q tf-keras

In [ ]:
import os
import sys

# PUNTO CRÍTICO 1 (continuación) — Forzar el motor legacy ANTES de importar TensorFlow
# Sin esta variable, Keras 3 intercepta todos los imports y rompe mrcnn
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import tensorflow as tf

# PUNTO CRÍTICO 2 — Deshabilitar Eager Execution
# Mask R-CNN construye un grafo estático (TF 1.x style); eager mode lo rompe
tf.compat.v1.disable_eager_execution()

print(f"TensorFlow version : {tf.__version__}")
print(f"Eager execution    : {tf.executing_eagerly()}")
print(f"TF_USE_LEGACY_KERAS: {os.environ.get('TF_USE_LEGACY_KERAS')}")

In [ ]:
import json
import numpy as np
import time
from PIL import Image, ImageDraw
import skimage.draw
import random

In [ ]:
!git clone https://github.com/DavidReveloLuna/MaskRCNN_Video.git

In [ ]:
cd MaskRCNN_Video/

In [ ]:
!python setup.py install

In [ ]:
ROOT_DIR = '/content/MaskRCNN_Video'
assert os.path.exists(ROOT_DIR), 'ROOT_DIR does not exist'

sys.path.append(ROOT_DIR)

# Estos imports ya respetan TF_USE_LEGACY_KERAS y la sesión sin eager
from mrcnn import visualize
from mrcnn.config import Config
from mrcnn import model as modellib, utils

In [ ]:
# Directory to save logs and trained model
MODEL_DIR = os.path.join(ROOT_DIR, "logs")

# Local path to trained weights file
COCO_MODEL_PATH = os.path.join(ROOT_DIR, "mask_rcnn_coco.h5")

# Download COCO trained weights from Releases if needed
if not os.path.exists(COCO_MODEL_PATH):
    utils.download_trained_weights(COCO_MODEL_PATH)

In [ ]:
class CustomConfig(Config):
    """Configuration for training on the blood cell dataset (RBC / WBC)."""
    NAME = "object"

    GPU_COUNT = 1
    IMAGES_PER_GPU = 1

    # background + RBC + WBC
    NUM_CLASSES = 1 + 2

    IMAGE_MIN_DIM = 512
    IMAGE_MAX_DIM = 512

    STEPS_PER_EPOCH = 500
    VALIDATION_STEPS = 5

    BACKBONE = 'resnet50'

    RPN_ANCHOR_SCALES = (8, 16, 32, 64, 128)
    TRAIN_ROIS_PER_IMAGE = 32
    MAX_GT_INSTANCES = 50
    POST_NMS_ROIS_INFERENCE = 500
    POST_NMS_ROIS_TRAINING = 1000

config = CustomConfig()
config.display()

In [ ]:
class CustomDataset(utils.Dataset):

    def load_custom(self, dataset_dir, subset):
        """Load a subset of the blood cell dataset.
        dataset_dir: Root directory of the dataset.
        subset: Subset to load: 'train' or 'val'
        """
        self.add_class("object", 1, "RBC")
        self.add_class("object", 2, "WBC")

        assert subset in ["train", "val"]
        dataset_dir = os.path.join(dataset_dir, subset)

        # VGG Image Annotator JSON format
        annotations1 = json.load(open(os.path.join(dataset_dir, "via_region_data.json")))
        annotations = list(annotations1.values())

        # Skip images without annotations
        annotations = [a for a in annotations if a['regions']]

        for a in annotations:
            polygons = [r['shape_attributes'] for r in a['regions'].values()]
            objects  = [s['region_attributes']['objetos'] for s in a['regions'].values()]

            name_dict = {"RBC": 1, "WBC": 2}
            num_ids = [name_dict[obj] for obj in objects]

            image_path = os.path.join(dataset_dir, a['filename'])
            image = skimage.io.imread(image_path)
            height, width = image.shape[:2]

            self.add_image(
                "object",
                image_id=a['filename'],
                path=image_path,
                width=width, height=height,
                polygons=polygons,
                num_ids=num_ids)

    def load_mask(self, image_id):
        """Generate instance masks for an image.
        Returns:
            masks    : bool array [height, width, instance_count]
            class_ids: 1D array of class IDs
        """
        image_info = self.image_info[image_id]
        if image_info["source"] != "object":
            return super(self.__class__, self).load_mask(image_id)

        info = self.image_info[image_id]
        num_ids = info['num_ids']
        mask = np.zeros(
            [info["height"], info["width"], len(info["polygons"])],
            dtype=np.uint8)

        for i, p in enumerate(info["polygons"]):
            rr, cc = skimage.draw.polygon(p['all_points_y'], p['all_points_x'])
            mask[rr, cc, i] = 1

        num_ids = np.array(num_ids, dtype=np.int32)
        return mask, num_ids

    def image_reference(self, image_id):
        """Return the path of the image."""
        info = self.image_info[image_id]
        if info["source"] == "object":
            return info["path"]
        # Bug fix: agregar return para que no devuelva None implícitamente
        return super(self.__class__, self).image_reference(image_id)

In [ ]:
dataset_train = CustomDataset()
dataset_train.load_custom("/content/MaskRCNN_Video/images", "train")
dataset_train.prepare()

# CORRECCIÓN: usar subset="val" para validación (el original usaba "train" por error)
dataset_val = CustomDataset()
dataset_val.load_custom("/content/MaskRCNN_Video/images", "val")
dataset_val.prepare()

In [ ]:
# Create model in training mode
model = modellib.MaskRCNN(mode="training", config=config, model_dir=MODEL_DIR)

In [ ]:
init_with = "coco"  # imagenet, coco, or last

if init_with == "imagenet":
    model.load_weights(model.get_imagenet_weights(), by_name=True)
elif init_with == "coco":
    model.load_weights(COCO_MODEL_PATH, by_name=True,
                       exclude=["mrcnn_class_logits", "mrcnn_bbox_fc",
                                "mrcnn_bbox", "mrcnn_mask"])
elif init_with == "last":
    model.load_weights(model.find_last(), by_name=True)

In [ ]:
# Load and display random samples
dataset = dataset_train
image_ids = np.random.choice(dataset.image_ids, 4)
for image_id in image_ids:
    print(image_id)
    image = dataset.load_image(image_id)
    mask, class_ids = dataset.load_mask(image_id)
    print(dataset.image_reference(image_id))
    visualize.display_top_masks(image, mask, class_ids, dataset.class_names)

In [ ]:
from mrcnn.model import log

image_id = random.choice(dataset.image_ids)
image = dataset.load_image(image_id)
mask, class_ids = dataset.load_mask(image_id)
bbox = utils.extract_bboxes(mask)

print("image_id ", image_id, dataset.image_reference(image_id))
log("image", image)
log("mask", mask)
log("class_ids", class_ids)
log("bbox", bbox)
visualize.display_instances(image, bbox, mask, class_ids, dataset.class_names)

## 🚀 Entrenamiento — PUNTO CRÍTICO 3

Debido a conflictos de memoria y de sesión de Keras en celdas interactivas, el entrenamiento se ejecuta mediante el script externo `train_mrcnn.py`.

Esto garantiza que la inicialización de variables de TF ocurra en un proceso limpio y aislado, evitando errores como `FailedPreconditionError: Attempting to use uninitialized value`.

In [ ]:
# Generar el script externo de entrenamiento
train_script = '''
import os
import sys

# PUNTO CRÍTICO 1: Motor Keras legacy
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import tensorflow as tf

# PUNTO CRÍTICO 2: Deshabilitar Eager Execution
tf.compat.v1.disable_eager_execution()

import json
import numpy as np
import skimage.draw

ROOT_DIR = '/content/MaskRCNN_Video'
sys.path.append(ROOT_DIR)

from mrcnn.config import Config
from mrcnn import model as modellib, utils

MODEL_DIR     = os.path.join(ROOT_DIR, "logs")
COCO_MODEL_PATH = os.path.join(ROOT_DIR, "mask_rcnn_coco.h5")


class CustomConfig(Config):
    NAME = "object"
    GPU_COUNT = 1
    IMAGES_PER_GPU = 1
    NUM_CLASSES = 1 + 2
    IMAGE_MIN_DIM = 512
    IMAGE_MAX_DIM = 512
    STEPS_PER_EPOCH = 500
    VALIDATION_STEPS = 5
    BACKBONE = 'resnet50'
    RPN_ANCHOR_SCALES = (8, 16, 32, 64, 128)
    TRAIN_ROIS_PER_IMAGE = 32
    MAX_GT_INSTANCES = 50
    POST_NMS_ROIS_INFERENCE = 500
    POST_NMS_ROIS_TRAINING = 1000


class CustomDataset(utils.Dataset):

    def load_custom(self, dataset_dir, subset):
        self.add_class("object", 1, "RBC")
        self.add_class("object", 2, "WBC")
        assert subset in ["train", "val"]
        dataset_dir = os.path.join(dataset_dir, subset)
        annotations1 = json.load(open(os.path.join(dataset_dir, "via_region_data.json")))
        annotations = [a for a in annotations1.values() if a["regions"]]
        for a in annotations:
            polygons = [r["shape_attributes"] for r in a["regions"].values()]
            objects  = [s["region_attributes"]["objetos"] for s in a["regions"].values()]
            name_dict = {"RBC": 1, "WBC": 2}
            num_ids = [name_dict[obj] for obj in objects]
            image_path = os.path.join(dataset_dir, a["filename"])
            image = skimage.io.imread(image_path)
            height, width = image.shape[:2]
            self.add_image(
                "object", image_id=a["filename"], path=image_path,
                width=width, height=height, polygons=polygons, num_ids=num_ids)

    def load_mask(self, image_id):
        info = self.image_info[image_id]
        if info["source"] != "object":
            return super(self.__class__, self).load_mask(image_id)
        num_ids = info["num_ids"]
        mask = np.zeros(
            [info["height"], info["width"], len(info["polygons"])], dtype=np.uint8)
        for i, p in enumerate(info["polygons"]):
            rr, cc = skimage.draw.polygon(p["all_points_y"], p["all_points_x"])
            mask[rr, cc, i] = 1
        return mask, np.array(num_ids, dtype=np.int32)

    def image_reference(self, image_id):
        info = self.image_info[image_id]
        if info["source"] == "object":
            return info["path"]
        return super(self.__class__, self).image_reference(image_id)


if __name__ == "__main__":
    config = CustomConfig()

    dataset_train = CustomDataset()
    dataset_train.load_custom("/content/MaskRCNN_Video/images", "train")
    dataset_train.prepare()

    dataset_val = CustomDataset()
    dataset_val.load_custom("/content/MaskRCNN_Video/images", "val")
    dataset_val.prepare()

    model = modellib.MaskRCNN(mode="training", config=config, model_dir=MODEL_DIR)

    model.load_weights(COCO_MODEL_PATH, by_name=True,
                       exclude=["mrcnn_class_logits", "mrcnn_bbox_fc",
                                "mrcnn_bbox", "mrcnn_mask"])

    model.train(dataset_train, dataset_val,
                learning_rate=0.001,
                epochs=10,
                layers="heads")

    print("Entrenamiento completado.")
'''

with open('/content/MaskRCNN_Video/train_mrcnn.py', 'w') as f:
    f.write(train_script)

print("Script train_mrcnn.py generado correctamente.")

In [ ]:
# PUNTO CRÍTICO 3 — Ejecutar entrenamiento como proceso externo aislado
# Esto evita conflictos de sesión de Keras que ocurren en celdas interactivas
!python /content/MaskRCNN_Video/train_mrcnn.py

## 🔍 Inferencia

In [ ]:
class InferenceConfig(CustomConfig):
    GPU_COUNT = 1
    IMAGES_PER_GPU = 1
    DETECTION_MIN_CONFIDENCE = 0.6

inference_config = InferenceConfig()

In [ ]:
# Recrear el modelo en modo inferencia
model = modellib.MaskRCNN(mode="inference", config=inference_config, model_dir=MODEL_DIR)

In [ ]:
# Cargar los pesos entrenados (último checkpoint guardado)
model_path = model.find_last()

assert model_path != "", "Proporciona la ruta a los pesos entrenados"
print("Cargando pesos desde:", model_path)
model.load_weights(model_path, by_name=True)

In [ ]:
import skimage
import matplotlib.pyplot as plt

real_test_dir = '/content/MaskRCNN_Video/images/train/'
image_paths = [
    os.path.join(real_test_dir, f)
    for f in os.listdir(real_test_dir)
    if os.path.splitext(f)[1].lower() in ['.png', '.jpg', '.jpeg']
]

for image_path in image_paths:
    img = skimage.io.imread(image_path)

    # Normalizar a RGB de 3 canales
    if img.ndim != 3:
        image = skimage.color.gray2rgb(img)
    elif img.shape[-1] == 4:
        image = img[..., :3]
    else:
        image = img

    print(image_path)
    img_arr = np.array(image)
    results = model.detect([img_arr], verbose=1)
    r = results[0]
    visualize.display_instances(
        img, r['rois'], r['masks'], r['class_ids'],
        dataset_val.class_names, r['scores'], figsize=(5, 5))